# Biomarker Discovery from MOFA + Random Forest

Objective:
- identify the MOFA factors most consistently important for subtype classification
- map those factors back to the original omics features
- derive candidate biomarkers across mRNA, miRNA, and DNA methylation


In [13]:
import pandas as pd
import numpy as np
import mofax as mfx
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    make_scorer,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
)

In [14]:
mrna_data_vsn_fs = pd.read_csv("../../data/feature_selection/selected_features_mrna_data_vsn.csv", index_col=0)
mirna_data_vsn_fs = pd.read_csv("../../data/feature_selection/selected_features_mirna_data_vsn.csv", index_col=0)
meth_data_fs = pd.read_csv("../../data/feature_selection/selected_features_meth_data.csv", index_col=0)

In [15]:
clinical_data = pd.read_csv("../../data/cleaned_data/clinical_cleaned.csv", index_col=0)
y = clinical_data["tumor_subtype"].map({"other_subtype": 1, "ductal_type": 0})

In [16]:
def load_mofa_factors(model_path, clinical_index):
    model = mfx.mofa_model(model_path)
    factors = model.get_factors(df=True)
    factors = factors.loc[clinical_index]
    return model, factors

In [17]:
model_path = "../../data/latent/mofa_trained_vsn_fs.hdf5"
mofa_model, X = load_mofa_factors(model_path, clinical_data.index)

In [18]:
print(X.shape)
display(X.head())

(176, 10)


,Factor1,Factor2,Factor3,Factor4,Factor5,Factor6,Factor7,Factor8,Factor9,Factor10
patient_id,,,,,,,,,,
TCGA-2J-AAB1,1.846789,1.404765,-0.073263,0.364550,-0.275643,-0.062900,0.066504,3.088176,-0.041426,0.142506
TCGA-2J-AAB4,0.413031,0.158443,0.482249,-1.621558,0.201344,0.099848,-0.214202,0.681693,0.557081,0.398764
TCGA-2J-AAB6,1.488815,1.811948,-3.019068,-1.567511,-0.968232,-1.909873,0.199540,1.980096,0.410115,-0.368314
TCGA-2J-AAB8,1.705274,-1.985239,0.602638,0.902741,0.203677,-1.370834,-0.178664,1.429240,0.033482,-0.473454
TCGA-2J-AAB9,-1.056744,-1.295832,-0.081001,-2.247182,0.089500,0.945770,-0.135600,2.755386,-0.182717,0.067042


In [19]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [2, 3, 4, 5],
    "min_samples_leaf": [5, 10, 15, 20],
    "class_weight": ["balanced", "balanced_subsample"],
}

scoring = {
    "balanced_accuracy": make_scorer(balanced_accuracy_score),
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "pr_auc_score": make_scorer(average_precision_score, response_method="predict_proba"),
}

In [20]:
outer_cv_splits = 5
inner_cv_splits = 5
seeds = list(range(30))

importance_rows = []
performance_rows = []

for seed in seeds:
    outer_cv = StratifiedKFold(n_splits=outer_cv_splits, shuffle=True, random_state=seed)

    for fold_id, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        inner_cv = StratifiedKFold(n_splits=inner_cv_splits, shuffle=True, random_state=seed)

        rf_model = RandomForestClassifier(random_state=seed, n_jobs=1)

        grid_search = GridSearchCV(
            estimator=rf_model,
            param_grid=param_grid,
            scoring=scoring,
            refit="pr_auc_score",
            cv=inner_cv,
            n_jobs=-1,
            return_train_score=False,
        )

        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_

        y_pred = best_model.predict(X_test)
        y_score = best_model.predict_proba(X_test)[:, 1]

        performance_rows.append({
            "seed": seed,
            "fold": fold_id,
            "best_params": str(grid_search.best_params_),
            "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "pr_auc": average_precision_score(y_test, y_score),
        })

        importances = best_model.feature_importances_

        for factor, importance in zip(X.columns, importances):
            importance_rows.append({
                "seed": seed,
                "fold": fold_id,
                "factor": factor,
                "importance": float(importance),
                "best_params": str(grid_search.best_params_),
            })

KeyboardInterrupt: 

In [ ]:
importance_df = pd.DataFrame(importance_rows)
performance_df = pd.DataFrame(performance_rows)

In [ ]:
display(importance_df.head())
display(performance_df.head())

,seed,fold,factor,coefficient,abs_coefficient,direction
0,0,1,Factor1,-0.182853,0.182853,ductal_type
1,0,1,Factor2,0.116193,0.116193,other_subtype
2,0,1,Factor3,-0.084974,0.084974,ductal_type
3,0,1,Factor4,0.150754,0.150754,other_subtype
4,0,1,Factor5,0.046272,0.046272,other_subtype


,seed,fold,best_params,balanced_accuracy,precision,recall,f1,pr_auc
0,0,1,{'svm__C': 0.01},0.800000,0.666667,0.666667,0.666667,0.600694
1,0,2,{'svm__C': 0.01},0.781609,0.571429,0.666667,0.615385,0.597050
2,0,3,{'svm__C': 1},0.643678,0.266667,0.666667,0.380952,0.553558
3,0,4,{'svm__C': 1},0.479885,0.142857,0.166667,0.153846,0.329166
4,0,5,{'svm__C': 0.1},0.800000,0.400000,0.800000,0.533333,0.774242


In [ ]:
factor_summary_global = (
    importance_df
    .groupby("factor", as_index=False)
    .agg(
        mean_importance=("importance", "mean"),
        median_importance=("importance", "median"),
        std_importance=("importance", "std"),
        n_models=("importance", "size"),
    )
    .sort_values("mean_importance", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
display(factor_summary_global)

,factor,mean_coefficient,mean_abs_coefficient,median_abs_coefficient,std_abs_coefficient,times_positive,times_negative,times_zero,n_models,stability,dominant_direction
0,Factor1,-0.542095,0.542095,0.583554,0.299238,0,150,0,150,1.000000,ductal_type
1,Factor7,-0.258669,0.265585,0.231862,0.245296,7,143,0,150,0.953333,ductal_type
2,Factor2,0.252211,0.252211,0.230763,0.150280,150,0,0,150,1.000000,other_subtype
3,Factor5,0.236217,0.250197,0.186213,0.209643,143,7,0,150,0.953333,other_subtype
4,Factor8,0.205161,0.205161,0.199779,0.059007,150,0,0,150,1.000000,other_subtype
5,Factor4,0.194166,0.194166,0.172921,0.094982,150,0,0,150,1.000000,other_subtype
6,Factor9,0.138533,0.138883,0.112158,0.105407,148,2,0,150,0.986667,other_subtype
7,Factor6,-0.130504,0.133489,0.118463,0.107127,12,138,0,150,0.920000,ductal_type
8,Factor10,0.068991,0.084146,0.057539,0.072226,118,32,0,150,0.786667,other_subtype
9,Factor3,-0.052010,0.082543,0.080173,0.034479,21,129,0,150,0.860000,ductal_type


In [ ]:
importance_df["rank_within_model"] = (
    importance_df
    .groupby(["seed", "fold"])["importance"]
    .rank(method="first", ascending=False)
)

In [ ]:
TOP_K = 5

top_factor_counts = (
    importance_df.loc[importance_df["rank_within_model"] <= TOP_K]
    .groupby("factor", as_index=False)
    .agg(times_in_top_k=("factor", "count"))
)

factor_summary_global = factor_summary_global.merge(top_factor_counts, on="factor", how="left")
factor_summary_global["times_in_top_k"] = factor_summary_global["times_in_top_k"].fillna(0).astype(int)
factor_summary_global["top_k_frequency"] = factor_summary_global["times_in_top_k"] / (30 * 5)

factor_summary_global = factor_summary_global.sort_values(
    ["mean_importance", "top_k_frequency"],
    ascending=[False, False]
).reset_index(drop=True)

,factor,mean_coefficient,mean_abs_coefficient,median_abs_coefficient,std_abs_coefficient,times_positive,times_negative,times_zero,n_models,stability,dominant_direction
0,Factor1,-0.542095,0.542095,0.583554,0.299238,0,150,0,150,1.000000,ductal_type
1,Factor7,-0.258669,0.265585,0.231862,0.245296,7,143,0,150,0.953333,ductal_type
2,Factor2,0.252211,0.252211,0.230763,0.150280,150,0,0,150,1.000000,other_subtype
3,Factor5,0.236217,0.250197,0.186213,0.209643,143,7,0,150,0.953333,other_subtype
4,Factor8,0.205161,0.205161,0.199779,0.059007,150,0,0,150,1.000000,other_subtype


In [ ]:
display(factor_summary_global)

['Factor1', 'Factor7', 'Factor2', 'Factor5', 'Factor8']

In [ ]:
ROBUST_TOP_K = 5
MIN_TOPK_FREQUENCY = 0.50

robust_factors = factor_summary_global.loc[
    factor_summary_global["top_k_frequency"] >= MIN_TOPK_FREQUENCY
].head(ROBUST_TOP_K).copy()

top_factors = robust_factors["factor"].tolist()

print("Robust factors:", top_factors)
display(robust_factors)

Extract MOFA weights

In [21]:
weights_df = mofa_model.get_weights(df=True)

display(weights_df.head())

,Factor1,Factor2,Factor3,Factor4,Factor5,Factor6,Factor7,Factor8,Factor9,Factor10
ENSG00000000003,0.005497,-0.002377,-0.002999,-0.003571,-3.503687e-06,0.002044,4.274295e-07,-9.660314e-08,-1.100453e-07,0.011566
ENSG00000000460,0.002917,-0.000004,-0.002635,0.002773,-4.711943e-07,-0.003890,1.053027e-06,9.209595e-08,-6.746733e-07,0.015695
ENSG00000000938,0.000016,-0.011756,0.004317,0.001013,-2.583346e-05,-0.003754,2.295939e-06,-2.380894e-07,1.243465e-06,0.014679
ENSG00000000971,0.001768,-0.008830,0.006107,-0.003898,-1.279115e-04,-0.006179,-3.249432e-05,1.696427e-07,-9.082739e-07,0.025126
ENSG00000001167,0.002268,-0.000002,0.000849,0.002648,8.389202e-06,-0.003648,-3.223921e-07,8.253413e-09,-6.551385e-07,0.016557


In [ ]:
weights_long = (
    weights_df
    .reset_index()
    .melt(
        id_vars=["index"],
        var_name="factor",
        value_name="weight"
    )
    .rename(columns={"index": "feature"})
)

weights_long["abs_weight"] = weights_long["weight"].abs()

display(weights_long.head())


,feature,factor,weight,abs_weight
0,ENSG00000000003,Factor1,0.005497,0.005497
1,ENSG00000000460,Factor1,0.002917,0.002917
2,ENSG00000000938,Factor1,0.000016,0.000016
3,ENSG00000000971,Factor1,0.001768,0.001768
4,ENSG00000001167,Factor1,0.002268,0.002268


In [25]:
mrna_features = set(mrna_data_vsn_fs.columns)
mirna_features = set(mirna_data_vsn_fs.columns)
meth_features = set(meth_data_fs.columns)

In [26]:
def assign_view(feature):
    if feature in mrna_features:
        return "mRNA"
    elif feature in mirna_features:
        return "miRNA"
    elif feature in meth_features:
        return "Methylation"
    else:
        return "Unknown"

In [27]:
weights_long['view'] = weights_long['feature'].apply(assign_view)

In [28]:
display(weights_long)

,feature,factor,weight,abs_weight,view
0,ENSG00000000003,Factor1,0.005497,0.005497,mRNA
1,ENSG00000000460,Factor1,0.002917,0.002917,mRNA
2,ENSG00000000938,Factor1,0.000016,0.000016,mRNA
3,ENSG00000000971,Factor1,0.001768,0.001768,mRNA
4,ENSG00000001167,Factor1,0.002268,0.002268,mRNA
...,...,...,...,...,...
709735,cg18851100,Factor10,-0.023025,0.023025,Methylation
709736,cg13213810,Factor10,-0.046587,0.046587,Methylation
709737,cg12502079,Factor10,-0.029857,0.029857,Methylation
709738,cg06422471,Factor10,-0.052069,0.052069,Methylation


Cruzar direção de SVM com pesos do MOFA

Agora vais combinar:

direção do fator no SVM
sinal do peso da feature no MOFA
Regra:

fator other_subtype + peso positivo -> feature apoia other_subtype

fator other_subtype + peso negativo -> feature apoia ductal_type

fator ductal_type + peso positivo -> feature apoia ductal_type

fator ductal_type + peso negativo -> feature apoia other_subtype

In [ ]:
TOP_FEATURES_PER_SIGN = 15

candidate_rows = []

for _, factor_row in robust_factors.iterrows():
    factor = factor_row["factor"]

    subset = weights_long.loc[weights_long["factor"] == factor].copy()

    for view_name, view_df in subset.groupby("view"):
        top_pos = view_df.sort_values("weight", ascending=False).head(TOP_FEATURES_PER_SIGN)
        top_neg = view_df.sort_values("weight", ascending=True).head(TOP_FEATURES_PER_SIGN)

        selected = pd.concat([top_pos, top_neg], axis=0).drop_duplicates()

        for _, row in selected.iterrows():
            candidate_rows.append({
                "factor": factor,
                "factor_mean_importance": factor_row["mean_importance"],
                "top_k_frequency": factor_row["top_k_frequency"],
                "view": row["view"],
                "feature": row["feature"],
                "weight": row["weight"],
                "abs_weight": row["abs_weight"],
            })

candidate_biomarkers = (
    pd.DataFrame(candidate_rows)
    .sort_values(
        ["factor_mean_importance", "factor", "view", "abs_weight"],
        ascending=[False, True, True, False]
    )
    .reset_index(drop=True)
)

candidate_biomarkers


In [30]:
display(candidate_biomarkers.head(20))

,factor,factor_direction,factor_mean_abs_coefficient,factor_stability,view,feature,weight,abs_weight,feature_direction
0,Factor1,ductal_type,0.542095,1.0,Methylation,cg10536999,-0.561907,0.561907,other_subtype
1,Factor1,ductal_type,0.542095,1.0,Methylation,cg12563935,0.535716,0.535716,ductal_type
2,Factor1,ductal_type,0.542095,1.0,Methylation,cg07700514,0.535569,0.535569,ductal_type
3,Factor1,ductal_type,0.542095,1.0,Methylation,cg04904331,0.534368,0.534368,ductal_type
4,Factor1,ductal_type,0.542095,1.0,Methylation,cg16389901,-0.529511,0.529511,other_subtype
5,Factor1,ductal_type,0.542095,1.0,Methylation,cg20151476,-0.524580,0.524580,other_subtype
6,Factor1,ductal_type,0.542095,1.0,Methylation,cg01893212,0.519085,0.519085,ductal_type
7,Factor1,ductal_type,0.542095,1.0,Methylation,cg20434926,-0.510976,0.510976,other_subtype
8,Factor1,ductal_type,0.542095,1.0,Methylation,cg02467990,0.509585,0.509585,ductal_type
9,Factor1,ductal_type,0.542095,1.0,Methylation,cg00582971,0.505884,0.505884,ductal_type


In [ ]:
candidate_biomarkers["rank_within_factor_view"] = (
    candidate_biomarkers
    .groupby(["factor", "view"])["abs_weight"]
    .rank(method="first", ascending=False)
)

candidate_biomarkers_final = candidate_biomarkers.sort_values(
    ["factor_mean_importance", "factor", "view", "rank_within_factor_view"],
    ascending=[False, True, True, True]
).reset_index(drop=True)



In [ ]:
display(candidate_biomarkers_final.head(30))

,factor,factor_direction,factor_mean_abs_coefficient,factor_stability,view,feature,weight,abs_weight,feature_direction,rank_within_factor_view
0,Factor1,ductal_type,0.542095,1.0,Methylation,cg10536999,-0.561907,0.561907,other_subtype,1.0
1,Factor1,ductal_type,0.542095,1.0,Methylation,cg12563935,0.535716,0.535716,ductal_type,2.0
2,Factor1,ductal_type,0.542095,1.0,Methylation,cg07700514,0.535569,0.535569,ductal_type,3.0
3,Factor1,ductal_type,0.542095,1.0,Methylation,cg04904331,0.534368,0.534368,ductal_type,4.0
4,Factor1,ductal_type,0.542095,1.0,Methylation,cg16389901,-0.529511,0.529511,other_subtype,5.0
5,Factor1,ductal_type,0.542095,1.0,Methylation,cg20151476,-0.524580,0.524580,other_subtype,6.0
6,Factor1,ductal_type,0.542095,1.0,Methylation,cg01893212,0.519085,0.519085,ductal_type,7.0
7,Factor1,ductal_type,0.542095,1.0,Methylation,cg20434926,-0.510976,0.510976,other_subtype,8.0
8,Factor1,ductal_type,0.542095,1.0,Methylation,cg02467990,0.509585,0.509585,ductal_type,9.0
9,Factor1,ductal_type,0.542095,1.0,Methylation,cg00582971,0.505884,0.505884,ductal_type,10.0
